# E21 — o número e a barra

Este caderno mede a conta mais velha do mundo: a média. Ela é a primeira coisa que se faz
com um dado, e a última em que se confia sem barra.

**A tentativa.** Olhar os últimos vinte e um pregões de um índice, contar em quantos deles
o índice subiu, e chamar o resultado de "o mundo". Depois olhar os últimos duzentos e
cinquenta e dois e chamar o novo resultado de "o mundo" também. Os dois números discordam, e
a aritmética não diz se a diferença é do mundo ou do tamanho da janela.

**O que se mede.**

1. a fração de dias de alta nas duas janelas, e a diferença entre elas;
2. a **barra** dessa fração --- de quanto ela se mexe sozinha --- contra a dispersão medida em
   milhares de mundos sorteados, para quatro tamanhos de janela;
3. a dispersão entre todas as janelas móveis da série real, contra a mesma barra.

In [1]:
# <- brinque com: SERIE, CURTA, LONGA, JANELAS, MUNDOS, SEMENTE
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, graficos, proporcao

RAIZ = Path.cwd()
SERIE = "sp500.csv"      # a série do arquivo do projeto anterior (.old/dados/)
CURTA = 21               # um mês de pregões
LONGA = 252              # um ano de pregões
JANELAS = (21, 63, 252, 1260)   # um mês, um trimestre, um ano, cinco anos
MUNDOS = 4000            # quantos mundos sorteados por tamanho de janela
SEMENTE = 21

precos = dados.carregar_serie(SERIE)
variacoes = proporcao.variacao(precos)
print("frevolab %s | %s: %d pregões, de %s a %s" % (
    frevolab.VERSAO, SERIE, len(precos),
    precos.index.min().date(), precos.index.max().date()))

frevolab 0.1.0 | sp500.csv: 6719 pregões, de 2000-01-03 a 2026-09-21


## A fração de altas, no mundo real

A leitura mais direta do dado: dos últimos vinte e um pregões, em quantos o índice subiu? E
dos últimos duzentos e cinquenta e dois? As duas respostas são a mesma pergunta, e não são o
mesmo número.

In [2]:
# As duas janelas do fim da série, e a dispersão entre TODAS as janelas de um mês.
fracao_curta = proporcao.fracao(variacoes.tail(CURTA))
fracao_longa = proporcao.fracao(variacoes.tail(LONGA))
janelas_curtas = proporcao.janelas(variacoes, CURTA)

barra_curta = proporcao.barra(fracao_longa, CURTA)
barra_longa = proporcao.barra(fracao_longa, LONGA)

print("janela de %d dias: %.2f%% de altas | janela de %d dias: %.2f%% | diferença: %.2f pontos" % (
    CURTA, 100 * fracao_curta, LONGA, 100 * fracao_longa, 100 * (fracao_curta - fracao_longa)))
print("a barra da janela curta: %.2f pontos | a da longa: %.2f pontos" % (
    100 * barra_curta, 100 * barra_longa))
print("a diferença vale %.2f barras da janela curta" % (
    (fracao_curta - fracao_longa) / barra_curta))
print()
print("janelas de %d dias na série inteira: %d | dispersão entre elas: %.2f pontos" % (
    CURTA, len(janelas_curtas), 100 * janelas_curtas.std(ddof=1)))
print("a barra prevista para uma janela de %d dias: %.2f pontos | razão: %.3f" % (
    CURTA, 100 * barra_curta, janelas_curtas.std(ddof=1) / barra_curta))
print("fora de duas barras: %.1f%% das janelas | mínima %.2f%% | máxima %.2f%%" % (
    100 * float((np.abs(janelas_curtas - fracao_longa) > 2 * barra_curta).mean()),
    100 * janelas_curtas.min(), 100 * janelas_curtas.max()))

janela de 21 dias: 42.86% de altas | janela de 252 dias: 54.76% | diferença: -11.90 pontos
a barra da janela curta: 10.86 pontos | a da longa: 3.14 pontos
a diferença vale -1.10 barras da janela curta

janelas de 21 dias na série inteira: 6698 | dispersão entre elas: 10.40 pontos
a barra prevista para uma janela de 21 dias: 10.86 pontos | razão: 0.957
fora de duas barras: 1.9% das janelas | mínima 19.05% | máxima 85.71%


## O mundo sorteado: a barra contra a dispersão medida

A barra é uma previsão, e uma previsão se confere. Aqui se sorteiam 4000 mundos para cada
tamanho de janela, todos com a **mesma** probabilidade de alta, e se mede a dispersão das
frações medidas contra a previsão sqrt(p(1-p)/n).

In [3]:
# A previsão contra a dispersão medida, para quatro tamanhos de janela.
sorteio = np.random.default_rng(SEMENTE)
p = fracao_longa
medido, previsto = {}, {}
for n in JANELAS:
    fracoes = proporcao.mundos(p, n, MUNDOS, sorteio)
    medido[n] = float(fracoes.std(ddof=1))
    previsto[n] = proporcao.barra(p, n)

print("%8s %14s %14s %10s" % ("dias", "previsto(%)", "medido(%)", "razão"))
for n in JANELAS:
    print("%8d %14.3f %14.3f %10.3f" % (n, 100 * previsto[n], 100 * medido[n],
                                        medido[n] / previsto[n]))
print()
print("a razão entre medido e previsto tem de ficar perto de um nas quatro linhas:")
print("a barra não é um enfeite, é uma conta que se pode conferir.")

    dias    previsto(%)      medido(%)      razão
      21         10.861         10.768      0.991
      63          6.271          6.250      0.997
     252          3.135          3.132      0.999
    1260          1.402          1.428      1.019

a razão entre medido e previsto tem de ficar perto de um nas quatro linhas:
a barra não é um enfeite, é uma conta que se pode conferir.


## As figuras

In [4]:
# Figura 1: a barra cai com a raiz do número de dias, e a dispersão medida cai com ela.
fig, eixo = plt.subplots(figsize=(8.6, 4.2))
eixo.loglog(JANELAS, [100 * previsto[n] for n in JANELAS], "o-", color="#1f4e79",
            label="a barra prevista: raiz de p(1-p)/n")
eixo.loglog(JANELAS, [100 * medido[n] for n in JANELAS], "s--", color="#b03a2e",
            label="a dispersão medida em %d mundos" % MUNDOS)
eixo.set_xlabel("dias na janela")
eixo.set_ylabel("dispersão da fração (pontos percentuais)")
eixo.set_title("A barra da média, prevista e medida")
eixo.grid(True, which="both", ls=":", lw=0.6, alpha=0.6)
eixo.legend()
graficos.salvar(fig, "E21_proporcao", 1)
plt.close(fig)

In [5]:
# Figura 2: as frações de um mês que a série de fato teve, contra a barra.
fig, eixo = plt.subplots(figsize=(8.6, 4.2))
eixo.hist(100 * janelas_curtas, bins=40, color="#1f4e79", alpha=0.85)
eixo.axvline(100 * fracao_longa, color="#333333", lw=1.4, label="a fração de %d dias" % LONGA)
for k in (1, 2):
    for lado in (-1, 1):
        eixo.axvline(100 * (fracao_longa + lado * k * barra_curta), color="#b03a2e",
                     ls="--" if k == 1 else ":", lw=1.2)
eixo.set_xlabel("fração de dias de alta em %d dias (%%)" % CURTA)
eixo.set_ylabel("quantas janelas")
eixo.set_title("As %d janelas de %d dias da série, contra a barra" % (len(janelas_curtas), CURTA))
eixo.plot([], [], color="#b03a2e", ls="--", label="uma barra")
eixo.plot([], [], color="#b03a2e", ls=":", label="duas barras")
eixo.legend()
graficos.salvar(fig, "E21_proporcao", 2)
plt.close(fig)

## Leitura visual das figuras

Feita nesta sessão abrindo os dois .png pela ponte de visão (AGENTS.md §9). O que segue é
observação, e observação não vira número.

**Figura 1.** Escala logarítmica nos dois eixos, quatro pontos por linha, de 21 a 1260 dias no
eixo x; o eixo y é logarítmico e traz as marcas 2, 3, 4, 6 e 10. As duas linhas caem juntas: no ponto de 21 dias as duas ficam em
torno de 10,8, e em 1260 as duas ficam em torno de 1,4. O que o eixo faz de propósito: em papel
logarítmico uma queda pela raiz vira reta de inclinação menos um meio, e é isso que se quer ver
--- as duas retas são paralelas. A separação entre as curvas é menor que a espessura do traço
em três dos quatro pontos, e o quarto ponto é onde o sorteio de 4000 mundos mais se afasta.

**Figura 2.** Histograma de 6698 janelas, eixo x de 20 a 85 por cento e eixo y até 1200. A
forma é de sino, com o corpo entre 35 e 75 e o pico perto de 55; a linha cheia (a fração dos
252 dias) cai dentro do corpo, e não numa ponta. As tracejadas ficam a uma barra dela e as
pontilhadas a duas. O eixo não engana aqui: os intervalos são iguais e as linhas de referência
estão declaradas na legenda. O que a figura mostra e a legenda não diz: as duas barras deixam
de fora quase todo o corpo de um lado só nas pontas --- as janelas extremas são poucas, mas
distantes, e a forma tem caudas mais longas do que um sino de mesma largura.

In [6]:
# O resultado: um objeto por grandeza, em português, para o livro citar por comando.
nome = {21: "vinte_um", 63: "sessenta_e_tres", 252: "duzentos_e_cinquenta_e_dois",
        1260: "mil_duzentos_e_sessenta"}
resultado = {
    "barra_serie_dias": int(len(precos)),
    "fracao_curta_dias": CURTA,
    "fracao_curta_pct": 100 * fracao_curta,
    "fracao_longa_dias": LONGA,
    "fracao_longa_pct": 100 * fracao_longa,
    "fracao_diferenca_pct": 100 * abs(fracao_curta - fracao_longa),
    "fracao_diferenca_barras": abs(fracao_curta - fracao_longa) / barra_curta,
    "barra_curta_pct": 100 * barra_curta,
    "barra_longa_pct": 100 * barra_longa,
    "mundos": MUNDOS,
    "janelas_curtas_conta": int(len(janelas_curtas)),
    "janelas_curtas_dispersao_pct": 100 * janelas_curtas.std(ddof=1),
    "janelas_curtas_razao": janelas_curtas.std(ddof=1) / barra_curta,
    "janelas_curtas_fora_pct": round(100 * float((np.abs(janelas_curtas - fracao_longa) > 2 * barra_curta).mean()), 1),
    "janelas_curtas_minima_pct": 100 * janelas_curtas.min(),
    "janelas_curtas_maxima_pct": 100 * janelas_curtas.max(),
}
for n in JANELAS:
    resultado["mundo_%s_previsto_pct" % nome[n]] = 100 * previsto[n]
    resultado["mundo_%s_medido_pct" % nome[n]] = 100 * medido[n]
    resultado["mundo_%s_razao" % nome[n]] = medido[n] / previsto[n]
    resultado["mundo_%s_dias" % nome[n]] = n

caminho = Path("lab/resultados/E21_proporcao.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))

lab/resultados/E21_proporcao.json gravado | 32 grandezas
